# Team Form Report

A supporter-focused view of a team's most recent completed matches in one Rugby Tracker competition and season.

Set `COMPETITION`, `SEASON`, `TEAM`, and `FORM_WINDOW` below, then run all cells. Competition and team names are matched case-insensitively. The form window defaults to five completed matches.

In [ ]:
COMPETITION = "PWR"
SEASON = "2025/26"
TEAM = "Harlequins Women"
FORM_WINDOW = 5


## Setup and data loading

In [ ]:
from __future__ import annotations

import html
import os
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_rows", 100)
plt.style.use("seaborn-v0_8-whitegrid")

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "rugby_tracker").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from within the RugbyTracker repository.")

if not isinstance(FORM_WINDOW, int) or isinstance(FORM_WINDOW, bool) or FORM_WINDOW < 1:
    raise ValueError("FORM_WINDOW must be a positive whole number.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DB_PATH = Path(os.environ.get("RUGBY_TRACKER_DB", REPO_ROOT / "data" / "rugbytracker.db")).expanduser()
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Rugby Tracker database not found: {DB_PATH}")

sys.path.insert(0, str(REPO_ROOT / "src"))
from rugby_tracker.standings import get_ruleset  # noqa: E402

connection = sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
print(f"Using database: {DB_PATH}")


In [ ]:
competition_sql = "SELECT id, name, season, gender, ruleset FROM competitions WHERE name = ? COLLATE NOCASE"
params = [COMPETITION.strip()]
if SEASON is not None:
    competition_sql += " AND season = ? COLLATE NOCASE"
    params.append(str(SEASON).strip())
competitions = [dict(row) for row in connection.execute(competition_sql, params)]
if not competitions:
    available = pd.read_sql_query("SELECT name, season FROM competitions ORDER BY name, season", connection)
    raise ValueError(
        f"Competition not found: {COMPETITION!r} (season={SEASON!r}). Available:\n"
        + available.to_string(index=False)
    )
if len(competitions) > 1:
    seasons = ", ".join(str(row["season"]) for row in competitions)
    raise ValueError(f"Competition {COMPETITION!r} exists in multiple seasons ({seasons}); set SEASON.")

competition = competitions[0]
if not competition["ruleset"]:
    raise ValueError(f"{competition['name']} {competition['season']} has no standings ruleset.")
ruleset = get_ruleset(competition["ruleset"])

matches_sql = """
SELECT m.*, h.name AS home_team_name, a.name AS away_team_name, v.name AS venue_name
FROM matches m
JOIN teams h ON h.id = m.home_team_id
JOIN teams a ON a.id = m.away_team_id
LEFT JOIN venues v ON v.id = m.venue_id
WHERE m.competition_id = ?
ORDER BY m.match_date, COALESCE(m.kickoff_time, ''), m.id
"""
all_matches = [dict(row) for row in connection.execute(matches_sql, (competition["id"],))]
team_names = sorted(
    {m["home_team_name"] for m in all_matches} | {m["away_team_name"] for m in all_matches},
    key=str.casefold,
)
matched_names = [name for name in team_names if name.casefold() == TEAM.strip().casefold()]
if not matched_names:
    raise ValueError(
        f"Team not found in {competition['name']} {competition['season']}: {TEAM!r}. "
        f"Available: {', '.join(team_names)}"
    )
team_name = matched_names[0]
completed_matches = [
    m for m in all_matches
    if m["home_score"] is not None and m["away_score"] is not None
]
team_matches = [
    m for m in completed_matches
    if team_name in (m["home_team_name"], m["away_team_name"])
]
if not team_matches:
    raise ValueError(f"No completed matches found for {team_name} in this competition.")

selected_matches = team_matches[-FORM_WINDOW:]
print(f"Available teams: {', '.join(team_names)}")


In [ ]:
KNOCKOUT_STAGES = {"quarter-final", "semi-final", "final"}

def competition_points(match: dict, is_home: bool) -> int | None:
    stage = str(match.get("round") or "").strip().casefold()
    if stage in ruleset.league_table.excluded_rounds:
        return None
    scored = int(match["home_score"] if is_home else match["away_score"])
    conceded = int(match["away_score"] if is_home else match["home_score"])
    tries = match["home_tries"] if is_home else match["away_tries"]
    scoring = ruleset.scoring
    points = scoring.win_points if scored > conceded else scoring.draw_points if scored == conceded else scoring.loss_points
    if tries is not None and int(tries) >= scoring.try_bonus_threshold:
        points += scoring.try_bonus_points
    if scored < conceded and conceded - scored <= scoring.losing_bonus_margin:
        points += scoring.losing_bonus_points
    return points

def team_view(match: dict) -> dict:
    is_home = match["home_team_name"] == team_name
    scored = int(match["home_score"] if is_home else match["away_score"])
    conceded = int(match["away_score"] if is_home else match["home_score"])
    tries_for = match["home_tries"] if is_home else match["away_tries"]
    tries_against = match["away_tries"] if is_home else match["home_tries"]
    result = "W" if scored > conceded else "L" if scored < conceded else "D"
    round_name = str(match["round"] or "—")
    knockout_stages = KNOCKOUT_STAGES | set(ruleset.league_table.excluded_rounds)
    is_knockout = round_name.strip().casefold() in knockout_stages
    margin = scored - conceded
    return {
        "Match ID": int(match["id"]),
        "Date": pd.to_datetime(match["match_date"]).date(),
        "Round / stage": round_name,
        "Match type": "Knockout" if is_knockout else "League",
        "Home / away": "Home" if is_home else "Away",
        "Opponent": match["away_team_name"] if is_home else match["home_team_name"],
        "Venue": match["venue_name"] or "—",
        "Final score": f"{scored}–{conceded}",
        "Result": result,
        "Points scored": scored,
        "Points conceded": conceded,
        "Tries scored": int(tries_for) if tries_for is not None else pd.NA,
        "Tries conceded": int(tries_against) if tries_against is not None else pd.NA,
        "Margin": margin,
        "Margin description": "Draw" if margin == 0 else f"Won by {margin}" if margin > 0 else f"Lost by {abs(margin)}",
        "Competition points": competition_points(match, is_home),
    }

season_results = pd.DataFrame(team_view(match) for match in team_matches)
form_results = pd.DataFrame(team_view(match) for match in selected_matches)

def summary(frame: pd.DataFrame) -> dict:
    played = len(frame)
    return {
        "Played": played,
        "Won": int((frame["Result"] == "W").sum()),
        "Drawn": int((frame["Result"] == "D").sum()),
        "Lost": int((frame["Result"] == "L").sum()),
        "Win percentage": float((frame["Result"] == "W").mean() * 100),
        "Points scored": int(frame["Points scored"].sum()),
        "Points conceded": int(frame["Points conceded"].sum()),
        "Points difference": int(frame["Margin"].sum()),
        "Average points scored": float(frame["Points scored"].mean()),
        "Average points conceded": float(frame["Points conceded"].mean()),
        "Points difference / match": float(frame["Margin"].mean()),
        "Average tries scored": float(frame["Tries scored"].mean()) if frame["Tries scored"].notna().any() else None,
        "Average tries conceded": float(frame["Tries conceded"].mean()) if frame["Tries conceded"].notna().any() else None,
    }

form_summary = summary(form_results)
season_summary = summary(season_results)

def cards(items):
    boxes = "".join(
        "<div style='padding:12px 16px;border:1px solid #bbb;border-radius:8px;min-width:125px'>"
        f"<small>{html.escape(str(label))}</small><br><b style='font-size:1.2em'>{html.escape(str(value))}</b></div>"
        for label, value in items
    )
    display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap;margin:8px 0 18px'>{boxes}</div>"))


## Report overview

**Form means the team's most recent completed matches**, ordered by match date, kick-off time (when stored), and match ID. Scheduled, postponed, abandoned, and incomplete fixtures do not contribute.

In [ ]:
date_start = form_results.iloc[0]["Date"]
date_end = form_results.iloc[-1]["Date"]
display(HTML(
    f"<h2>{html.escape(team_name)}</h2>"
    f"<p><b>{html.escape(competition['name'])} · {html.escape(str(competition['season']))}</b></p>"
))
cards([
    ("Requested window", FORM_WINDOW),
    ("Completed matches available", len(season_results)),
    ("Matches included", len(form_results)),
    ("Date range", f"{date_start:%d %b %Y} – {date_end:%d %b %Y}"),
])
if len(form_results) < FORM_WINDOW:
    display(HTML(
        f"<p><i>Only {len(form_results)} completed match(es) are available, so all have been included.</i></p>"
    ))


## Current form summary

In [ ]:
try_cards = []
if form_results["Tries scored"].notna().any():
    try_cards = [
        ("Tries scored", int(form_results["Tries scored"].sum(min_count=1))),
        ("Tries conceded", int(form_results["Tries conceded"].sum(min_count=1))),
    ]
cards([
    ("Played", form_summary["Played"]),
    ("Won", form_summary["Won"]),
    ("Drawn", form_summary["Drawn"]),
    ("Lost", form_summary["Lost"]),
    ("Win percentage", f"{form_summary['Win percentage']:.1f}%"),
])
cards([
    ("Points scored", form_summary["Points scored"]),
    ("Points conceded", form_summary["Points conceded"]),
    ("Points difference", f"{form_summary['Points difference']:+d}"),
    ("Avg scored / match", f"{form_summary['Average points scored']:.1f}"),
    ("Avg conceded / match", f"{form_summary['Average points conceded']:.1f}"),
    *try_cards,
])
applicable_points = form_results["Competition points"].dropna()
if not applicable_points.empty:
    cards([("Competition points earned", int(applicable_points.sum()))])
    if len(applicable_points) < len(form_results):
        display(HTML("<p><small>Knockout matches excluded by the competition ruleset do not receive league competition points.</small></p>"))


## Form sequence

Oldest → most recent. Letters remain visible so the sequence does not depend on colour.

In [ ]:
result_colours = {"W": "#d8f3dc", "D": "#eeeeee", "L": "#ffe0e0"}
sequence = " ".join(
    f"<span style='display:inline-block;padding:7px 11px;margin:2px;border:1px solid #777;"
    f"border-radius:5px;background:{result_colours[result]};font-weight:bold'>{result}</span>"
    for result in form_results["Result"]
)
display(HTML(f"<div style='font-size:1.15em'>{sequence}</div>"))


## Recent results

Most recent first. Knockout stages are labelled explicitly; competition points are shown as “—” when the ruleset excludes that stage.

In [ ]:
recent_columns = [
    "Date", "Round / stage", "Match type", "Home / away", "Opponent", "Venue",
    "Final score", "Result", "Points scored", "Points conceded",
    "Tries scored", "Tries conceded", "Margin description", "Competition points",
]
recent_results = form_results[recent_columns].iloc[::-1].copy()
recent_results["Competition points"] = recent_results["Competition points"].map(
    lambda value: "—" if pd.isna(value) else int(value)
)
display(recent_results.style.hide(axis="index"))


## Results breakdown and scoring form

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
counts = form_results["Result"].value_counts().reindex(["W", "D", "L"], fill_value=0)
bars = axes[0].bar(["Wins", "Draws", "Losses"], counts, color=["#4c78a8", "#9e9e9e", "#e45756"])
axes[0].set(title="Recent result counts", xlabel="Result", ylabel="Matches", ylim=(0, max(1, counts.max() + 1)))
axes[0].bar_label(bars, padding=3)

x = range(1, len(form_results) + 1)
labels = [f"{row['Opponent']}\n{row['Date']:%d %b}" for _, row in form_results.iterrows()]
axes[1].plot(x, form_results["Points scored"], marker="o", linewidth=2, label="Points scored")
axes[1].plot(x, form_results["Points conceded"], marker="s", linestyle="--", linewidth=2, label="Points conceded")
axes[1].set(title="Scoring form (oldest → most recent)", xlabel="Opponent and date", ylabel="Points", xticks=list(x), xticklabels=labels)
axes[1].legend()
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


## Try-scoring form

In [ ]:
complete_try_rows = form_results.dropna(subset=["Tries scored", "Tries conceded"])
if complete_try_rows.empty:
    display(HTML("<p><i>Try data is not available for these matches, so no try-scoring chart is shown.</i></p>"))
else:
    if len(complete_try_rows) < len(form_results):
        display(HTML("<p><i>Only matches with recorded try data are shown.</i></p>"))
    x = range(1, len(complete_try_rows) + 1)
    labels = [f"{row['Opponent']}\n{row['Date']:%d %b}" for _, row in complete_try_rows.iterrows()]
    width = 0.38
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.bar([value - width / 2 for value in x], complete_try_rows["Tries scored"], width, label="Tries scored")
    ax.bar([value + width / 2 for value in x], complete_try_rows["Tries conceded"], width, label="Tries conceded")
    ax.set(title="Try-scoring form (oldest → most recent)", xlabel="Opponent and date", ylabel="Tries", xticks=list(x), xticklabels=labels)
    ax.legend()
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()


## Home and away form

Match counts are included because a short form window can produce very small home or away samples.

In [ ]:
def split_record(location: str) -> dict:
    subset = form_results[form_results["Home / away"] == location]
    played = len(subset)
    return {
        "Venue status": location,
        "Played": played,
        "Won": int((subset["Result"] == "W").sum()),
        "Drawn": int((subset["Result"] == "D").sum()),
        "Lost": int((subset["Result"] == "L").sum()),
        "Points scored": int(subset["Points scored"].sum()),
        "Points conceded": int(subset["Points conceded"].sum()),
        "Points difference": int(subset["Margin"].sum()),
        "Win percentage": f"{(subset['Result'] == 'W').mean() * 100:.1f}%" if played else "—",
    }

home_away = pd.DataFrame([split_record("Home"), split_record("Away")])
display(home_away.style.hide(axis="index"))
display(HTML("<p><small>These are descriptive counts; avoid strong conclusions where either sample is small.</small></p>"))


## Current streak

The streak uses the full completed-match sequence for the selected competition and season, so it may extend beyond the chosen form window.

In [ ]:
latest_result = season_results.iloc[-1]["Result"]
streak_rows = []
for index in range(len(season_results) - 1, -1, -1):
    row = season_results.iloc[index]
    if row["Result"] != latest_result:
        break
    streak_rows.append(row)
streak_rows.reverse()
streak_words = {"W": "winning", "D": "drawing", "L": "losing"}
cards([
    ("Streak type", streak_words[latest_result].title()),
    ("Matches", len(streak_rows)),
    ("Began", f"{streak_rows[0]['Date']:%d %b %Y}"),
    ("Opponents", ", ".join(row["Opponent"] for row in streak_rows)),
])


## Comparison with season performance

“Better” and “worse” use a simple **10 percentage-point win-rate threshold**; differences inside that range are “broadly similar”. This is descriptive, not statistically definitive.

In [ ]:
win_rate_gap = form_summary["Win percentage"] - season_summary["Win percentage"]
comparison_label = "Better than season average" if win_rate_gap >= 10 else "Worse than season average" if win_rate_gap <= -10 else "Broadly similar to season average"

def optional_number(value):
    return "—" if value is None else f"{value:.1f}"

comparison = pd.DataFrame([
    ("Win percentage", f"{form_summary['Win percentage']:.1f}%", f"{season_summary['Win percentage']:.1f}%"),
    ("Average points scored", f"{form_summary['Average points scored']:.1f}", f"{season_summary['Average points scored']:.1f}"),
    ("Average points conceded", f"{form_summary['Average points conceded']:.1f}", f"{season_summary['Average points conceded']:.1f}"),
    ("Average tries scored", optional_number(form_summary["Average tries scored"]), optional_number(season_summary["Average tries scored"])),
    ("Points difference / match", f"{form_summary['Points difference / match']:+.1f}", f"{season_summary['Points difference / match']:+.1f}"),
], columns=["Measure", "Recent form", "Season to date"]).set_index("Measure")
cards([("Recent-results assessment", comparison_label), ("Win-rate difference", f"{win_rate_gap:+.1f} percentage points")])
display(comparison)


## Form trend

To keep the conclusion explainable, the selected matches are split chronologically into an earlier and later group. With at least four matches, a change of **5 points in average points difference per match** is labelled improving or declining; a smaller change is broadly stable. The figures—not the label—are the important output.

In [ ]:
if len(form_results) < 4:
    trend_label = "Insufficient data"
    display(HTML("<p><b>Insufficient data:</b> at least four completed matches are needed for the split-period trend.</p>"))
else:
    split = len(form_results) // 2
    earlier = form_results.iloc[:split]
    later = form_results.iloc[split:]
    earlier_summary = summary(earlier)
    later_summary = summary(later)
    pd_change = later_summary["Points difference / match"] - earlier_summary["Points difference / match"]
    trend_label = "Improving" if pd_change >= 5 else "Declining" if pd_change <= -5 else "Broadly stable"
    trend = pd.DataFrame([
        ("Matches", len(earlier), len(later), len(later) - len(earlier)),
        ("Win percentage", earlier_summary["Win percentage"], later_summary["Win percentage"], later_summary["Win percentage"] - earlier_summary["Win percentage"]),
        ("Average points scored", earlier_summary["Average points scored"], later_summary["Average points scored"], later_summary["Average points scored"] - earlier_summary["Average points scored"]),
        ("Average points conceded", earlier_summary["Average points conceded"], later_summary["Average points conceded"], later_summary["Average points conceded"] - earlier_summary["Average points conceded"]),
        ("Points difference / match", earlier_summary["Points difference / match"], later_summary["Points difference / match"], pd_change),
    ], columns=["Measure", "Earlier matches", "Later matches", "Change"]).set_index("Measure")
    cards([("Form trend", trend_label), ("Avg points-difference change", f"{pd_change:+.1f} per match")])
    display(trend.style.format("{:.1f}", subset=["Earlier matches", "Later matches", "Change"]))


---

## Interpretation and experimental notes

This report describes what happened; it does not predict the next result. A short run can be shaped by opposition strength, home/away balance, injuries, international call-ups, conditions, fixture gaps, and knockout matches. No mathematical adjustment is made for those factors.

Prototype decisions to revisit before application integration:

- Keep five as the default, while retaining a configurable positive whole-number window.
- Validate whether supporters find the split-period trend label useful; the underlying comparison should remain visible.
- Consider optional league-only form where competitions mix league and knockout stages.
- Retain home/away counts so small samples are obvious.
- Review which charts add enough value for the future application report.
- Move reusable team-relative match, competition-points, streak, and summary calculations into shared application code if this prototype is adopted.

The notebook is designed for standard Jupyter PDF export. Tables and charts use compact widths and visible text labels.